# 10 - Mejoras: entrenamiento con `class_weight`

Este experimento entrena una ResNet18 nueva usando todas las lesiones disponibles fuera del test anterior y pondera la pérdida para compensar el desbalance, en lugar de submuestrear los nevos. El checkpoint se guarda como `models/cnn_resnet18_class_weight.pt`; no reemplaza `models/cnn_resnet18.pt`, que sigue siendo el modelo de inferencia actual.

**Importante:** el test anterior se excluye por `lesion_id` tanto del entrenamiento como de la validación y no se evalúa aquí. Esta validación es exploratoria: no constituye una evaluación independiente ni debe usarse para afirmar superioridad sobre el modelo anterior. Ejecutar el notebook con el entorno/kernel del proyecto; en CPU puede tardar varias horas.

## 1. Preparar datos y entorno

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.experimento_class_weight import (
    UMBRALES,
    UMBRAL_RECALL,
    entrenar_modelo,
    metricas,
    preparar_particiones,
    seleccionar_umbral,
)
from src.utils import SEED

RUTA_METADATA = ROOT / "data" / "raw" / "HAM10000_metadata.csv"
RUTA_TEST_ANTERIOR = ROOT / "data" / "processed" / "test.csv"
RUTA_CHECKPOINT = ROOT / "models" / "cnn_resnet18_class_weight.pt"
RUTA_REPORTE = ROOT / "reports" / "class_weight_experiment.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Dispositivo: {device}")
print(f"Checkpoint nuevo: {RUTA_CHECKPOINT}")

Dispositivo: cpu
Checkpoint nuevo: c:\Trabajo final\models\cnn_resnet18_class_weight.pt


## 2. Partición sin las lesiones del test anterior

Se conserva la prevalencia natural de melanoma en train y validación. `pos_weight` se calcula con las cantidades de la partición de entrenamiento.

In [2]:
train_df, val_df = preparar_particiones(RUTA_METADATA, RUTA_TEST_ANTERIOR)
test_lesions = set(pd.read_csv(RUTA_TEST_ANTERIOR)["lesion_id"])

assert set(train_df["lesion_id"]).isdisjoint(test_lesions)
assert set(val_df["lesion_id"]).isdisjoint(test_lesions)
assert set(train_df["lesion_id"]).isdisjoint(val_df["lesion_id"])

n_mel = int(train_df["etiqueta"].sum())
n_nevo = int((train_df["etiqueta"] == 0).sum())
pos_weight = n_nevo / n_mel
print(f"Train: {len(train_df)} imágenes ({n_mel} melanomas, {n_nevo} nevos)")
print(f"Validación: {len(val_df)} imágenes; prevalencia melanoma {val_df['etiqueta'].mean():.1%}")
print(f"Lesiones excluidas del test anterior: {len(test_lesions)}")
print(f"pos_weight: {pos_weight:.3f}")

Train: 4616 imágenes (393 melanomas, 4223 nevos)
Validación: 1155 imágenes; prevalencia melanoma 8.5%
Lesiones excluidas del test anterior: 246
pos_weight: 10.746


## 3. Entrenar la CNN ponderada

El mejor checkpoint de validación se escribe de manera atómica durante el entrenamiento. Si el proceso se interrumpe, el modelo anterior y el último mejor checkpoint de esta variante permanecen separados.

In [3]:
salida = entrenar_modelo(
    train_df,
    val_df,
    ponderar_clases=True,
    device=device,
    checkpoint_path=RUTA_CHECKPOINT,
)
print(f"Checkpoint guardado: {RUTA_CHECKPOINT.exists()}")

[ponderado F1] época 1/8: train=0.8990, val=0.8613
[ponderado F1] época 2/8: train=0.7228, val=0.8106
[ponderado F1] época 3/8: train=0.6775, val=0.6688
[ponderado F1] época 4/8: train=0.6558, val=0.6708
[ponderado F1] época 5/8: train=0.6572, val=0.6564
[ponderado F1] época 6/8: train=0.6403, val=0.6695
[ponderado F1] época 7/8: train=0.6294, val=0.6808
[ponderado F1] época 8/8: train=0.6485, val=0.6593
[ponderado F2] época 1/12: train=0.7043, val=0.6596
[ponderado F2] época 2/12: train=0.5513, val=0.6034
[ponderado F2] época 3/12: train=0.4761, val=0.5732
[ponderado F2] época 4/12: train=0.4700, val=0.6446
[ponderado F2] época 5/12: train=0.4776, val=0.5396
[ponderado F2] época 6/12: train=0.4057, val=0.6180
[ponderado F2] época 7/12: train=0.3780, val=0.6156
[ponderado F2] época 8/12: train=0.3739, val=0.5734
[ponderado F2] época 9/12: train=0.3515, val=0.6271
Parada temprana.
Checkpoint guardado: True


## 4. Evaluación exploratoria en validación

El umbral se elige con la misma regla declarada: entre la grilla de umbrales, maximizar F1 macro sujeto a recall de melanoma ≥ 0,85. También se reporta el resultado con umbral 0,5.

In [4]:
metricas_05 = metricas(salida["y_true"], salida["probabilidades"], 0.5)
umbral = seleccionar_umbral(salida["y_true"], salida["probabilidades"])
metricas_seleccionadas = (
    metricas(salida["y_true"], salida["probabilidades"], umbral)
    if umbral is not None
    else None
)

reporte = {
    "experimento": "ResNet18 con class_weight",
    "evaluacion": "validacion exploratoria; se excluyo el test anterior",
    "semilla": SEED,
    "test_anterior_excluido": True,
    "tamano_test_anterior_excluido": len(test_lesions),
    "entrenamiento": {
        "imagenes": len(train_df),
        "melanomas": n_mel,
        "nevos": n_nevo,
        "pos_weight": salida["pos_weight"],
    },
    "validacion": {
        "imagenes": len(val_df),
        "prevalencia_melanoma": float(val_df["etiqueta"].mean()),
        "metricas_umbral_0_5": metricas_05,
        "metricas_umbral_seleccionado": metricas_seleccionadas,
    },
    "regla_umbral": {
        "grilla": list(UMBRALES),
        "recall_minimo": UMBRAL_RECALL,
        "criterio": "mayor F1 macro entre umbrales elegibles",
    },
    "historial": salida["historial"],
    "checkpoint": str(RUTA_CHECKPOINT.relative_to(ROOT)),
}

RUTA_REPORTE.parent.mkdir(parents=True, exist_ok=True)
RUTA_REPORTE.write_text(
    json.dumps(reporte, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(f"Reporte guardado en {RUTA_REPORTE}")
pd.DataFrame([metricas_05, metricas_seleccionadas] if metricas_seleccionadas else [metricas_05], index=["umbral 0.5", f"umbral elegido ({umbral})"] if metricas_seleccionadas else ["umbral 0.5"])

Reporte guardado en c:\Trabajo final\reports\class_weight_experiment.json


,umbral,accuracy,precision_melanoma,recall_melanoma,especificidad,ppv_estimado_prevalencia_10pct,f1_macro,falsos_positivos,falsos_negativos,matriz_confusion
umbral 0.5,0.5,0.862338,0.369099,0.877551,0.860927,0.412149,0.719647,147,12,"[[910, 147], [12, 86]]"
umbral elegido (0.5),0.5,0.862338,0.369099,0.877551,0.860927,0.412149,0.719647,147,12,"[[910, 147], [12, 86]]"
